In [ ]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# PATHS (set here when running via main_pipeline; standalone fallback below)
try:
    spikeglx_folder
    base_folder
except NameError:
    spikeglx_folder = Path(r'C:\Users\labuser\Ilaria\Project\small_rec\2142_g0\2142_g0_imec0')
    base_folder     = Path(r'C:\Users\labuser\Ilaria\Project\processed\2142_tutorial_output')
base_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# SPIKEGLX-SPECIFIC PARAMETERS
stream_name = 'imec0.ap'
freq_min    = 400.          # highpass cutoff (Hz), default for Neuropixels

try:
    sorter = SORTER_KS
except NameError:
    sorter = 'kilosort4'

cref_operator  = 'median'
cref_reference = 'global'

# Fallback linear probe pitch (um)
contact_pitch_um = 20.0

# Shared variables exposed to main_pipeline
_source_path     = str(spikeglx_folder)
_analyzer_sparse = True

In [ ]:
# LOAD
raw_rec = si.read_spikeglx(spikeglx_folder, stream_name=stream_name, load_sync_channel=False)
print(raw_rec)

In [ ]:
# PROBE SETUP
# Attaches a fallback single-column linear probe if no geometry is present.
# contact_pitch_um presets:
#   20 um  —  Neuropixels 1.0 inner column  [default]
#   25 um  —  Neuropixels 2.0
#   50 um  —  Cambridge Neurotech H-series
#  100 um  —  generic low-density silicon probe

if raw_rec.get_property("contact_vector") is None:
    from probeinterface import Probe

    n = raw_rec.get_num_channels()
    probe = Probe(ndim=2, si_units="um")
    probe.set_contacts(
        positions    = np.column_stack([np.zeros(n), np.arange(n, dtype=float) * contact_pitch_um]),
        shapes       = "circle",
        shape_params = {"radius": min(contact_pitch_um * 0.35, 7.5)},
    )
    probe.create_auto_shape(probe_type="tip")
    probe.set_device_channel_indices(np.arange(n))
    raw_rec = raw_rec.set_probe(probe, group_mode="by_probe")
    print(f"[probe] Fallback linear probe attached — {n} ch | {contact_pitch_um} um pitch | {(n-1)*contact_pitch_um:.0f} um span")
else:
    print(f"[probe] Geometry already present — {raw_rec.get_num_channels()} ch, using as-is.")

si.plot_probe_map(raw_rec, with_channel_ids=False)
plt.tight_layout()
plt.show()

In [ ]:
# PREPROCESSING
_n_raw_channels = raw_rec.get_num_channels()

if manual_channel_ids is not None:
    raw_rec = raw_rec.channel_slice(manual_channel_ids)
    print(f"Manual selection: kept {len(manual_channel_ids)} channels")

rec1 = si.phase_shift(raw_rec)
rec1 = si.highpass_filter(raw_rec, freq_min=freq_min)

if artifact_timestamps_s:
    _fs       = raw_rec.get_sampling_frequency()
    _triggers = [np.array([int(t * _fs) for t in artifact_timestamps_s], dtype=np.int64)]
    rec1 = si.remove_artifacts(rec1, list_triggers=_triggers,
                                ms_before=artifact_ms_before, ms_after=artifact_ms_after,
                                mode='zeros')
    print(f"Artifact removal: blanked {len(artifact_timestamps_s)} event(s)")

bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)

if extra_bad_channel_ids:
    bad_channel_ids = list(set(bad_channel_ids.tolist()) | set(extra_bad_channel_ids))

rec2 = rec1.remove_channels(bad_channel_ids)
rec3 = si.phase_shift(rec2)
rec  = si.common_reference(rec3, operator=cref_operator, reference=cref_reference)
print(rec)